# Async Context Manager, `search_batch()`, and Startup Validation

This notebook covers three features introduced in Medha **0.4.3**:

1. **Async context manager** — `async with Medha(...) as m:` replaces manual `start()` / `close()` calls
2. **`search_batch()`** — embed and search multiple questions in a single round-trip
3. **Performance comparison** — 5 sequential `search()` calls vs one `search_batch()` call
4. **Startup validation** — `start()` now probes the backend and surfaces connectivity errors early

**Requirements:** `pip install "medha-archai[fastembed]" pandas`  
All cells use `InMemoryBackend` — no external services needed.

In [ ]:
import time

import pandas as pd

from medha import Medha, Settings
from medha.backends.memory import InMemoryBackend
from medha.embeddings.fastembed_adapter import FastEmbedAdapter
from medha.exceptions import StorageError
from medha.interfaces.storage import VectorStorageBackend
from medha.types import CacheEntry, CacheResult

# Shared embedder — loaded once, reused across all sections
embedder = FastEmbedAdapter(
    model_name="BAAI/bge-small-en-v1.5",
    cache_dir="./.fastembed_cache",
)

## 1. Async Context Manager

Starting with **0.4.3**, Medha implements the async context manager protocol.
Use `async with Medha(...) as m:` instead of calling `start()` / `close()` manually.
The context manager guarantees cleanup even if an exception occurs inside the block.

```python
# Before 0.4.3
m = Medha("my_cache", embedder=embedder, settings=settings)
await m.start()
try:
    ...
finally:
    await m.close()

# From 0.4.3 onwards
async with Medha("my_cache", embedder=embedder, settings=settings) as m:
    ...
# close() is called automatically — even if an exception is raised
```

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("ctx_demo", embedder=embedder, settings=settings) as m:
    # Store three question-query pairs
    await m.store("How many users are there?",  "SELECT COUNT(*) FROM users")
    await m.store("List all active products",    "SELECT * FROM products WHERE active = true")
    await m.store("Total revenue last month",    "SELECT SUM(amount) FROM orders WHERE MONTH(created_at) = MONTH(NOW()) - 1")

    # Search — the same question will get an L1_CACHE hit because store() populates L1
    hit = await m.search("How many users are there?")
    print(f"Strategy  : {hit.strategy.value}")
    print(f"Confidence: {hit.confidence:.3f}")
    print(f"Query     : {hit.generated_query}")
    print()

    # Clear L1 so the next search goes through vector tiers
    await m.clear_caches()
    hit2 = await m.search("user count")
    print(f"Paraphrase search — strategy: {hit2.strategy.value}")
    if hit2.generated_query:
        print(f"Matched query: {hit2.generated_query}")

print("\n→ Context exited: backend.close() was called automatically")

## 2. `search_batch()`

`search_batch()` embeds all questions in a **single `aembed_batch()` call**, then runs
the waterfall search for each question concurrently via `asyncio.gather`.
Use it when you need to look up many questions at once — for example, warming a test
harness, running a benchmark, or processing a batch of user queries in a web handler.

The results are returned in the **same order** as the input questions.

In [ ]:
PAIRS = [
    ("How many users?",         "SELECT COUNT(*) FROM users"),
    ("List all products",        "SELECT * FROM products"),
    ("Total revenue",            "SELECT SUM(amount) FROM orders"),
    ("Active sessions today",    "SELECT COUNT(*) FROM sessions WHERE DATE(started_at) = CURDATE()"),
    ("Average order value",      "SELECT AVG(amount) FROM orders"),
]

settings = Settings(backend_type="memory")

async with Medha("batch_demo", embedder=embedder, settings=settings) as m:
    for question, query in PAIRS:
        await m.store(question, query)

    questions = [q for q, _ in PAIRS]
    results = await m.search_batch(questions)

    rows = [
        {
            "question": q,
            "strategy": r.strategy.value,
            "confidence": round(r.confidence or 0.0, 3),
            "generated_query": r.generated_query or "",
        }
        for q, r in zip(questions, results)
    ]

df = pd.DataFrame(rows)
try:
    display(df)
except NameError:
    print(df.to_string(index=False))

## 3. Performance: Sequential vs Batch

The key difference:

| Approach | Embedding calls | Vector searches |
|---|---|---|
| 5 × `search()` | 5 × `aembed(text)` | 5 sequential |
| `search_batch([…])` | 1 × `aembed_batch(texts)` | 5 concurrent |

With a **remote embedder** (OpenAI API, a GPU server) the saving is dramatic:
batch cuts 4 network round-trips. With local FastEmbed the gain is smaller but
still visible because ONNX Runtime processes a full batch more efficiently than
N single-item calls.

The comparison below clears L1 and the embedding cache before each trial so
both paths pay the full embedding cost.

In [ ]:
BENCH_PAIRS = [
    ("Count registered users",   "SELECT COUNT(*) FROM users"),
    ("Revenue this quarter",      "SELECT SUM(amount) FROM orders WHERE QUARTER(created_at) = QUARTER(NOW())"),
    ("Top 5 products by sales",   "SELECT product_id, SUM(qty) AS sold FROM order_items GROUP BY product_id ORDER BY sold DESC LIMIT 5"),
    ("Pending support tickets",   "SELECT COUNT(*) FROM tickets WHERE status = 'pending'"),
    ("Monthly active users",      "SELECT COUNT(DISTINCT user_id) FROM events WHERE DATE_TRUNC('month', ts) = DATE_TRUNC('month', NOW())"),
]
BENCH_QUESTIONS = [q for q, _ in BENCH_PAIRS]

settings = Settings(backend_type="memory")
N_TRIALS = 5

async with Medha("perf_demo", embedder=embedder, settings=settings) as m:
    for question, query in BENCH_PAIRS:
        await m.store(question, query)

    seq_times = []
    batch_times = []

    for _ in range(N_TRIALS):
        # --- Sequential ---
        await m.clear_caches()  # flush L1 + embedding cache
        t0 = time.perf_counter()
        for q in BENCH_QUESTIONS:
            await m.search(q)
        seq_times.append((time.perf_counter() - t0) * 1000)

        # --- Batch ---
        await m.clear_caches()
        t0 = time.perf_counter()
        await m.search_batch(BENCH_QUESTIONS)
        batch_times.append((time.perf_counter() - t0) * 1000)

seq_median  = sorted(seq_times)[N_TRIALS // 2]
batch_median = sorted(batch_times)[N_TRIALS // 2]

print(f"5 × search()   median: {seq_median:6.1f} ms  ({seq_median / 5:.1f} ms each)")
print(f"search_batch() median: {batch_median:6.1f} ms")
print(f"Speedup               : {seq_median / batch_median:.2f}x")
print()
print("Note: with a remote embedder the speedup is proportionally much larger.")

## 4. Startup Validation

`start()` now probes the backend with a `count()` call before returning.
This surfaces connectivity issues early — at startup — rather than letting them
surface as cryptic errors in the middle of a search or store operation.

```python
# validate_on_start=True (default): start() raises StorageError if the backend
# is unreachable.  Fail fast, log clearly, fix the connection string.
settings = Settings(validate_on_start=True)

# validate_on_start=False: skip the probe.  Use this in unit tests or CI
# environments where the backend may not be reachable, or when you're using
# a mock backend that doesn't implement count().
settings = Settings(validate_on_start=False)
```

The cell below simulates an unreachable backend to show both paths.

In [ ]:
class UnreachableBackend(VectorStorageBackend):
    """Minimal stub that simulates a backend whose connectivity check fails."""

    async def initialize(self, collection_name, dimension, **kw): pass

    async def count(self, collection_name):
        # The legacy-collection probe (name ends with '_templates') is caught
        # by start() already.  Raise StorageError there so it is silently
        # swallowed.  Raise ConnectionError for the main collection so it
        # propagates and gets wrapped into StorageError by start().
        if collection_name.endswith("_templates"):
            raise StorageError("collection not found")
        raise ConnectionError("connection refused: 127.0.0.1:6333")

    async def search(self, collection_name, vector, limit=5, score_threshold=0.0): return []
    async def upsert(self, collection_name, entries): pass
    async def scroll(self, collection_name, limit=100, offset=None, with_vectors=False): return [], None
    async def delete(self, collection_name, ids): pass
    async def find_expired(self, collection_name): return []
    async def search_by_normalized_question(self, collection_name, normalized_question): return None
    async def find_by_query_hash(self, collection_name, query_hash): return []
    async def find_by_template_id(self, collection_name, template_id): return []
    async def drop_collection(self, collection_name): pass
    async def update_feedback(self, collection_name, point_id, correct): return 0
    async def close(self): pass


# --- Path A: validate_on_start=True (default) ---
print("=" * 60)
print("Path A: validate_on_start=True")
print("=" * 60)

m = Medha(
    "probe_test",
    embedder=embedder,
    backend=UnreachableBackend(),
    settings=Settings(validate_on_start=True),
)
try:
    await m.start()
    print("ERROR: start() should have raised!")
except StorageError as exc:
    print(f"StorageError caught (expected):")
    print(f"  {exc}")

print()

# --- Path B: validate_on_start=False ---
print("=" * 60)
print("Path B: validate_on_start=False")
print("=" * 60)

m2 = Medha(
    "probe_test",
    embedder=embedder,
    backend=UnreachableBackend(),
    settings=Settings(validate_on_start=False),
)
await m2.start()   # no exception — probe skipped
print("start() returned without raising (probe skipped)")
await m2.close()

print()
print("Recommendation: keep validate_on_start=True in production.")
print("Use validate_on_start=False only in unit tests or CI pipelines")
print("where the backend is intentionally absent.")